# Custom estimator comparison

Research question: how can a lab bring a prototype estimator into the same benchmark report as bundled estimators?

This notebook registers the example variance-ratio estimator and compares it with the bundled RS estimator on a small ground-truth design.

In [ ]:
from pathlib import Path

import pandas as pd

from lrdbench.defaults import build_default_estimator_registry
from lrdbench.examples.custom_estimator import build_variance_ratio_estimator
from lrdbench.manifest import manifest_from_mapping
from lrdbench.output_contract import validate_output_contract
from lrdbench.runner import BenchmarkRunner

In [ ]:
estimators = build_default_estimator_registry()
estimators.register("VarianceRatio", build_variance_ratio_estimator)

In [ ]:
manifest_data = {
    "manifest_id": "workshop_custom_estimator_v1",
    "name": "workshop_custom_estimator",
    "mode": "ground_truth",
    "source": {
        "type": "generator_grid",
        "generators": [
            {"family": "fGn", "params": {"H": [0.5, 0.7], "n": [256], "sigma": [1.0]}, "replicates": 1}
        ],
    },
    "estimators": [
        {"name": "RS", "family": "time_domain", "target_estimand": "hurst_scaling_proxy", "supports_ci": False},
        {
            "name": "VarianceRatio",
            "family": "external",
            "target_estimand": "hurst_scaling_proxy",
            "assumptions": ["finite_variance", "example_only"],
            "supports_ci": False,
            "supports_diagnostics": True,
            "params": {"min_n": 32},
        },
    ],
    "metrics": ["bias", "mae", "validity_rate", "runtime"],
    "leaderboards": [
        {
            "name": "estimator_comparison",
            "mode": "ground_truth",
            "component_metrics": ["mae", "validity_rate", "runtime"],
            "weights": {"mae": 0.5, "validity_rate": 0.3, "runtime": 0.2},
            "ranking_rule": "weighted_rank",
            "tie_break_rule": "mae",
        }
    ],
    "report": {"formats": ["html", "csv"], "export_root": "reports/notebooks/custom_estimator"},
    "seeds": {"global_seed": 20260427},
    "execution": {"max_workers": 1},
}

In [ ]:
manifest = manifest_from_mapping(manifest_data)
out = BenchmarkRunner(estimators=estimators).run(manifest, base_dir=Path.cwd())
errors = validate_output_contract(out.result_store_path)
assert errors == []
out.result_store_path

In [ ]:
result_store = Path(out.result_store_path)
metadata = pd.read_csv(result_store / "tables" / "estimator_metadata.csv")
leaderboard = pd.read_csv(result_store / "tables" / "leaderboard.csv")
metadata[["estimator_name", "family", "target_estimand"]], leaderboard[["estimator_name", "rank", "score"]]